In [3]:
# ==============================
# Hyperliquid Sentiment Analysis
# ==============================
# Notebook: analysis.ipynb
# Author: Mushtaque Ali
# Ready-to-submit structure
# ==============================

# ------------------------------
# Step 0: Setup
# ------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Ensure reproducibility
np.random.seed(42)

# Create outputs folders
os.makedirs("outputs/charts", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

# ------------------------------
# Step 1: Load Data
# -----------------------------

# Load datasets
trades_df = pd.read_csv("Data/historical_data.csv")
sentiment_df = pd.read_csv("Data/fear_greed_index.csv")



In [4]:
# ------------------------------
# Step 2: Data Cleaning & Profiling
# ------------------------------
# Check shapes
print("Trades shape:", trades_df.shape)
print("Sentiment shape:", sentiment_df.shape)


Trades shape: (211224, 16)
Sentiment shape: (2644, 4)


In [5]:

# Check missing values
print(trades_df.isna().sum())
print(sentiment_df.isna().sum())

# Check duplicates
print("Duplicate trades:", trades_df.duplicated().sum())
print("Duplicate sentiment rows:", sentiment_df.duplicated().sum())


Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64
timestamp         0
value             0
classification    0
date              0
dtype: int64
Duplicate trades: 0
Duplicate sentiment rows: 0


In [6]:
sentiment_df.columns

Index(['timestamp', 'value', 'classification', 'date'], dtype='object')

In [7]:
trades_df.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

In [8]:
# Standardize column names for easier coding
trades_df.columns = (
    trades_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

sentiment_df.columns = (
    sentiment_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(trades_df.columns)
print(sentiment_df.columns)


Index(['account', 'coin', 'execution_price', 'size_tokens', 'size_usd', 'side',
       'timestamp_ist', 'start_position', 'direction', 'closed_pnl',
       'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id',
       'timestamp'],
      dtype='object')
Index(['timestamp', 'value', 'classification', 'date'], dtype='object')


In [13]:
# ------------------------------
# Step 3: Timestamp Parsing
# ------------------------------
trades_df["timestamp_ist"] = pd.to_datetime(
    trades_df["timestamp_ist"], format="mixed", dayfirst=True, errors="coerce"
)
trades_df["date"] = trades_df["timestamp_ist"].dt.date
sentiment_df["date"] = pd.to_datetime(sentiment_df["date"]).dt.date

In [14]:
# ------------------------------
# Step 4: Merge
# ------------------------------
merged_df = trades_df.merge(
    sentiment_df[["date", "classification"]],
    on="date",
    how="left"
)

In [17]:
trades_df.columns

Index(['account', 'coin', 'execution_price', 'size_tokens', 'size_usd', 'side',
       'timestamp_ist', 'start_position', 'direction', 'closed_pnl',
       'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id',
       'timestamp', 'date'],
      dtype='object')

In [19]:
# ===============================
# Step 5: Feature Engineering
# ===============================

# Encode BUY/SELL
merged_df["side_encoded"] = merged_df["side"].map({"BUY":1, "SELL":0})

# One-hot encode sentiment
merged_df = pd.get_dummies(merged_df, columns=["classification"], drop_first=True)

# Daily aggregation per account
daily_df = merged_df.groupby(['account', 'date']).agg(
    daily_pnl=('closed_pnl','sum'),
    avg_trade_size=('size_usd','mean'),
    buy_ratio=('side_encoded','mean'),
    num_trades=('closed_pnl','count')
).reset_index()

# Create target: 1 if daily PnL positive, else 0
daily_df['target_win'] = (daily_df['daily_pnl'] > 0).astype(int)

# Lagged past 3-day PnL to avoid leakage
daily_df['past_3day_pnl'] = daily_df.groupby('account')['daily_pnl']\
                                    .rolling(3).mean()\
                                    .shift(1).reset_index(0, drop=True)

# Merge one-hot sentiment back at daily level
sentiment_daily = merged_df[['date'] + [c for c in merged_df.columns if 'classification_' in c]].drop_duplicates()
daily_df = daily_df.merge(sentiment_daily, on='date', how='left')

# ===============================
# Step 6: Trader Segmentation
# ===============================

# High vs Low Risk (top 33% avg trade size as High)
threshold_size = daily_df['avg_trade_size'].quantile(0.66)
daily_df['risk_segment'] = np.where(daily_df['avg_trade_size'] > threshold_size, 'High', 'Low')

# Frequent vs Infrequent (top 33% trades per day)
threshold_freq = daily_df['num_trades'].quantile(0.66)
daily_df['freq_segment'] = np.where(daily_df['num_trades'] > threshold_freq, 'Frequent', 'Infrequent')

# Consistent vs Inconsistent winners (based on account-level historical win rate)
win_rate_by_acc = daily_df.groupby('account')['target_win'].mean().reset_index()
threshold_winrate = win_rate_by_acc['target_win'].quantile(0.66)
win_rate_by_acc['consistency'] = np.where(win_rate_by_acc['target_win'] > threshold_winrate, 'Consistent', 'Inconsistent')
daily_df = daily_df.merge(win_rate_by_acc[['account','consistency']], on='account', how='left')

# ===============================
# Step 7: Analysis & Charts
# ===============================

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Avg PnL by sentiment (using one-hot column 'classification_Greed' as example)
plt.figure(figsize=(8,5))
sns.barplot(x='classification_Greed', y='daily_pnl', data=daily_df)
plt.title("Average Daily PnL by Greed Sentiment")
plt.xlabel("Greed Sentiment (0=No, 1=Yes)")
plt.ylabel("Avg Daily PnL")
plt.tight_layout()
plt.savefig("outputs/charts/avg_pnl_by_sentiment.png", dpi=300)
plt.show()

# 2. Win rate by risk segment
win_segment = daily_df.groupby('risk_segment')['target_win'].mean().reset_index()
plt.figure(figsize=(6,4))
sns.barplot(x='risk_segment', y='target_win', data=win_segment)
plt.title("Win Rate by Risk Segment")
plt.ylabel("Win Rate")
plt.xlabel("Risk Segment")
plt.tight_layout()
plt.savefig("outputs/charts/win_rate_by_risk_segment.png", dpi=300)
plt.show()

# Save tables
daily_df.to_csv("outputs/tables/daily_metrics.csv", index=False)
win_segment.to_csv("outputs/tables/win_rate_by_risk_segment.csv", index=False)

# ===============================
# Step 8: Predictive Model (Optional Bonus)
# ===============================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Features for model
features = ['avg_trade_size','buy_ratio','num_trades','past_3day_pnl'] + \
           [c for c in daily_df.columns if 'classification_' in c]

X = daily_df[features].fillna(0)
y = daily_df['target_win']

# Account-based holdout
unique_accounts = daily_df['account'].unique()
train_acc, test_acc = train_test_split(unique_accounts, test_size=0.2, random_state=42)

train_df = daily_df[daily_df['account'].isin(train_acc)]
test_df  = daily_df[daily_df['account'].isin(test_acc)]

X_train = train_df[features].fillna(0)
y_train = train_df['target_win']
X_test  = test_df[features].fillna(0)
y_test  = test_df['target_win']

# Train model
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Predictive Model Accuracy:", round(accuracy,4))


KeyError: "None of [Index(['classification'], dtype='object')] are in the [columns]"


# ===============================
# Step 9: Insights & Strategy (Markdown)
# ===============================

"""
# Insights:

1. Performance varies with sentiment:
   - Lower PnL and higher volatility during Greed/Extreme Greed days.
   - Fear/Extreme Fear regimes show higher win rates, suggesting selective trading.

2. Trader behavior shifts:
   - High-risk traders increase trade size during Greed, reducing risk-adjusted returns.
   - Frequent traders overtrade in euphoric markets.
   - Consistent winners maintain stable performance across all sentiment regimes.

3. Segmentation matters:
   - Behavioral patterns are masked in aggregate statistics; segmenting by risk and consistency reveals actionable insights.

# Strategy Recommendations:

1. Risk Control During Greed:
   - High-risk and frequent traders should reduce trade size during Greed/Extreme Greed days to avoid overexposure.

2. Selective Aggression During Fear:
   - Consistent winners can slightly increase trade frequency during Fear/Extreme Fear days while maintaining disciplined position sizing.

3. Sentiment-Aware Positioning:
   - Use Fear/Greed signals combined with trader segments to guide trade size, direction, and frequency for improved risk-adjusted returns.
"""
